# TOOLBOXLAP — Colab + Ollama + Optional Network Tunnel

One-cell launcher for Ollama + Hugging Face GGUF/Ollama models. In a Google-managed Colab runtime, the API stays local. When the same notebook is connected to a **local runtime you control**, it can optionally expose the API with ngrok.

**Important:** Google's current Colab FAQ prohibits offering unrelated web services and connecting to remote proxies from managed Colab runtimes. This notebook does not attempt to bypass those restrictions. The ngrok path is only enabled when the kernel is outside Colab's managed runtime.

**Website:** https://toolboxlap.com  |  **YouTube:** https://www.youtube.com/@TOOLBOXLAP-u1c  |  **GitHub:** https://github.com/toolboxlap-ve/TOOLBOXLAP-Colab-Ollama

In [ ]:
# TOOLBOXLAP — one-cell launcher
import os, sys, time, re, shutil, subprocess, threading
from urllib.parse import urlparse, unquote

DEFAULT_MODEL = 'hf.co/HauhauCS/Qwen3.5-9B-Uncensored-HauhauCS-Aggressive:Q4_K_M'
PUBLIC_MODEL_ID = 'toolboxlap'
OLLAMA_URL = 'http://127.0.0.1:11434'
PROXY_PORT = 5000
HIGH_CONTEXT = 131072
FALLBACK_CONTEXT = 65536
IN_COLAB_MANAGED = bool(os.environ.get('COLAB_RELEASE_TAG'))

print('=' * 72)
print('TOOLBOXLAP — Colab / Ollama / Hugging Face')
print('=' * 72)
print('Runtime:', 'Google Colab managed runtime' if IN_COLAB_MANAGED else 'Local / self-controlled runtime')

model = input(f'\nModel [ENTER = default: {DEFAULT_MODEL}]: ').strip() or DEFAULT_MODEL
def run(cmd, check=True, env=None):
    print('+', ' '.join(cmd), flush=True)
    return subprocess.run(cmd, check=check, text=True, env=env)

def pip_import(module, package=None):
    try: __import__(module)
    except ImportError: run([sys.executable,'-m','pip','install','-q',package or module])

pip_import('requests')
pip_import('flask')
if shutil.which('zstd') is None and shutil.which('apt-get'):
    run(['apt-get','update'])
    run(['apt-get','install','-y','zstd'])
if shutil.which('ollama') is None:
    run(['bash','-lc','curl -fsSL https://ollama.com/install.sh | sh'])
if shutil.which('ollama') is None: raise RuntimeError('Ollama installation failed.')

import requests
def wait_http(url, timeout=120):
    end=time.monotonic()+timeout
    while time.monotonic()<end:
        try:
            r=requests.get(url,timeout=3)
            if r.ok: return
        except Exception: pass
        time.sleep(1)
    raise TimeoutError(f'Timed out waiting for {url}')

def start_ollama(ctx):
    env=os.environ.copy()
    env.update({'OLLAMA_FLASH_ATTENTION':'1','OLLAMA_KV_CACHE_TYPE':'q8_0','OLLAMA_NUM_PARALLEL':'1','OLLAMA_MAX_LOADED_MODELS':'1','OLLAMA_KEEP_ALIVE':'30m','OLLAMA_CONTEXT_LENGTH':str(ctx)})
    p=subprocess.Popen(['ollama','serve'],env=env,stdout=subprocess.DEVNULL,stderr=subprocess.STDOUT)
    wait_http(OLLAMA_URL+'/api/tags')
    return p

ctx=HIGH_CONTEXT
ollama_proc=start_ollama(ctx)
try:
    run(['ollama','pull',model])
except Exception:
    try: ollama_proc.terminate()
    except Exception: pass
    ctx=FALLBACK_CONTEXT
    print('Retrying with fallback context 65536...')
    ollama_proc=start_ollama(ctx)
    run(['ollama','pull',model])

from flask import Flask, request, Response, jsonify
app=Flask(__name__)
@app.get('/health')
def health(): return jsonify({'ok':True,'model':PUBLIC_MODEL_ID,'backend':model,'context':ctx})
@app.get('/v1/models')
def models(): return jsonify({'object':'list','data':[{'id':PUBLIC_MODEL_ID,'object':'model','owned_by':'toolboxlap'}]})
@app.post('/v1/chat/completions')
def chat():
    body=request.get_json(force=True,silent=True) or {}
    body['model']=model
    if 'max_tokens' not in body and 'max_completion_tokens' not in body: body['max_tokens']=32768
    body.pop('max_completion_tokens',None)
    for k in ['reasoning_effort','tool_choice','parallel_tool_calls','store','metadata','service_tier','logprobs','top_logprobs','modalities','audio']: body.pop(k,None)
    r=requests.post(OLLAMA_URL+'/v1/chat/completions',json=body,stream=body.get('stream',False),timeout=600)
    if body.get('stream',False): return Response(r.iter_content(chunk_size=None),status=r.status_code,content_type=r.headers.get('content-type','text/event-stream'))
    return Response(r.content,status=r.status_code,content_type=r.headers.get('content-type','application/json'))

def serve(): app.run(host='0.0.0.0' if not IN_COLAB_MANAGED else '127.0.0.1',port=PROXY_PORT,debug=False,use_reloader=False)
threading.Thread(target=serve,daemon=True).start()
wait_http(f'http://127.0.0.1:{PROXY_PORT}/health')
print('\n✅ TOOLBOXLAP API READY')
print(f'Local Base URL: http://127.0.0.1:{PROXY_PORT}/v1')
print(f'Model ID: {PUBLIC_MODEL_ID}')
print(f'Backend model: {model}')
print(f'Context: {ctx}')
r=requests.get(f'http://127.0.0.1:{PROXY_PORT}/v1/models',timeout=15)
print(f'Local API test HTTP {r.status_code}')
print(r.text)

if IN_COLAB_MANAGED:
    print('\nℹ️ Managed Colab detected: public tunneling is disabled in this notebook.')
    print('To demonstrate the public-network version, connect Colab to a local runtime you control and run this same cell there.')
else:
    print('\n🌐 Local/self-controlled runtime detected.')
    pip_import('pyngrok','pyngrok')
    from getpass import getpass
    token=getpass('ngrok authtoken (input hidden): ').strip()
    if not token: raise RuntimeError('An ngrok authtoken is required for the public tunnel.')
    from pyngrok import ngrok
    ngrok.set_auth_token(token)
    tunnel=ngrok.connect(PROXY_PORT,'http')
    public_url=tunnel.public_url.rstrip('/')
    base_url=public_url+'/v1'
    print('\n✅ PUBLIC API READY')
    print('COPY THIS BASE URL INTO CLINE:')
    print(base_url)
    print(f'Cline Model ID: {PUBLIC_MODEL_ID}')
    print('Custom Header: ngrok-skip-browser-warning = true')
    print('Backend model:',model)
    print('Active context:',ctx)
    test=requests.get(public_url+'/v1/models',headers={'ngrok-skip-browser-warning':'true'},timeout=20)
    print('Public API test HTTP',test.status_code)
    print(test.text)
    print('✅ EVERYTHING IS WORKING')